####Setup

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


spark.conf.set(
    "spark.sql.session.timeZone",
   "UTC"
)


raw_root = "s3://l1l2bucket/data/raw/mudrex"

catalog = "crypto_lakehouse"
bronze_schema = "bronze"

print("Raw data:", raw_root)
print(
    "Bronze schema:",
    f"{catalog}.{bronze_schema}"
)

display(
    dbutils.fs.ls(raw_root)
)

Raw data: s3://l1l2bucket/data/raw/mudrex
Bronze schema: crypto_lakehouse.bronze


path,name,size,modificationTime
s3://l1l2bucket/data/raw/mudrex/deposits/,deposits/,0,1790609505528
s3://l1l2bucket/data/raw/mudrex/earn_history/,earn_history/,0,1790609505528
s3://l1l2bucket/data/raw/mudrex/ingestion_manifest/,ingestion_manifest/,0,1790609505528
s3://l1l2bucket/data/raw/mudrex/inr_spot_trades/,inr_spot_trades/,0,1790609505528
s3://l1l2bucket/data/raw/mudrex/usdt_fund_transactions/,usdt_fund_transactions/,0,1790609505528
s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/,usdt_spot_trades/,0,1790609505528
s3://l1l2bucket/data/raw/mudrex/withdrawals/,withdrawals/,0,1790609505528


####Mudrex USDT trades

In [0]:
usdt_spot_schema = StructType([
    StructField(
        "Investment Product",
        StringType(),
        True
    ),
    StructField(
        "Date (IST)",
        StringType(),
        True
    ),
    StructField(
        "Coin Name",
        StringType(),
        True
    ),
    StructField(
        "Symbol",
        StringType(),
        True
    ),
    StructField(
        "Transaction Type",
        StringType(),
        True
    ),
    StructField(
        "Coin Quantity",
        DecimalType(38, 18),
        True
    ),
    StructField(
        "Amount (USDT)",
        DecimalType(38, 18),
        True
    )
])


usdt_spot_raw_df = (
    spark.read
    .schema(usdt_spot_schema)
    .option("multiline", True)
    .option("recursiveFileLookup", True)
    .json(
        f"{raw_root}/usdt_spot_trades/"
    )
)


usdt_spot_bronze_df = (
    usdt_spot_raw_df
    .select(
        col("Investment Product")
        .alias("investment_product"),

        col("Date (IST)")
        .alias("date_ist"),

        col("Coin Name")
        .alias("coin_name"),

        col("Symbol")
        .alias("symbol"),

        col("Transaction Type")
        .alias("transaction_type"),

        col("Coin Quantity")
        .alias("coin_quantity"),

        col("Amount (USDT)")
        .alias("amount_usdt"),

        col("_metadata.file_name")
        .alias("source_file_name"),

        col("_metadata.file_path")
        .alias("_source_file")
    )
    .withColumn(
        "exchange",
        lit("Mudrex")
    )
    .withColumn(
        "source_type",
        lit("USDT Spot Trade")
    )
    .withColumn(
        "ingested_at",
        current_timestamp()
    )
    .dropDuplicates([
        "date_ist",
        "symbol",
        "transaction_type",
        "coin_quantity",
        "amount_usdt"
    ])
)


print(
    "Mudrex USDT Spot rows:",
    usdt_spot_bronze_df.count()
)

display(
    usdt_spot_bronze_df
    .orderBy(
        to_timestamp(
            col("date_ist"),
            "MMMM d, yyyy HH:mm:ss"
        ).desc()
    )
)

Mudrex USDT Spot rows: 37


investment_product,date_ist,coin_name,symbol,transaction_type,coin_quantity,amount_usdt,source_file_name,_source_file,exchange,source_type,ingested_at
Coins,"March 13, 2026 19:30:08",Sui,SUI,BUY,4.051362309902464000,4.250756183027779000,usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,Mudrex,USDT Spot Trade,2026-09-28T15:32:00.938Z
Coins,"March 13, 2026 19:25:20",Ethereum,ETH,BUY,0.003157137965290000,6.918202010536473000,usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,Mudrex,USDT Spot Trade,2026-09-28T15:32:00.938Z
Coins,"March 13, 2026 19:24:35",Bitcoin,BTC,BUY,0.000094112602467000,6.918202010536473000,usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,Mudrex,USDT Spot Trade,2026-09-28T15:32:00.938Z
Coins,"March 13, 2026 19:22:45",Solana,SOL,BUY,0.034687108373958000,3.193016312555295000,usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,Mudrex,USDT Spot Trade,2026-09-28T15:32:00.938Z
Coins,"March 13, 2026 19:19:35",Ondo,ONDO,BUY,38.452899654224275000,10.643387708517650000,usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,Mudrex,USDT Spot Trade,2026-09-28T15:32:00.938Z
Coins,"March 13, 2026 19:18:46",ChainLink,LINK,BUY,1.110548702356832000,10.643387708517650000,usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,Mudrex,USDT Spot Trade,2026-09-28T15:32:00.938Z
Coins,"February 18, 2026 16:56:14",Ethereum,ETH,BUY,0.010561198489765000,20.965068072972755000,usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,Mudrex,USDT Spot Trade,2026-09-28T15:32:00.938Z
Coins,"February 18, 2026 16:55:32",Solana,SOL,BUY,0.150411197024386000,12.580927982981102000,usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,Mudrex,USDT Spot Trade,2026-09-28T15:32:00.938Z
Coins,"February 18, 2026 16:54:16",Bitcoin,BTC,BUY,0.000233088573040000,15.726159978726375000,usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,Mudrex,USDT Spot Trade,2026-09-28T15:32:00.938Z
Coins,"February 18, 2026 16:53:24",Sui,SUI,BUY,5.400166140717734000,5.242053326242125000,usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,s3://l1l2bucket/data/raw/mudrex/usdt_spot_trades/usdt_spot_trades_20260927T153313_896563Z_ba766db5.json,Mudrex,USDT Spot Trade,2026-09-28T15:32:00.938Z


In [0]:
(
    usdt_spot_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        True
    )
    .saveAsTable(
        f"{catalog}.{bronze_schema}.mudrex_usdt_spot_trades"
    )
)


print(
    "Saved Mudrex USDT Spot rows:",
    spark.table(
        f"{catalog}.{bronze_schema}.mudrex_usdt_spot_trades"
    ).count()
)

Saved Mudrex USDT Spot rows: 37


#### Mudrex INR trades

In [0]:
inr_spot_schema = StructType([
    StructField(
        "Investment Product",
        StringType(),
        True
    ),
    StructField(
        "Date (IST)",
        StringType(),
        True
    ),
    StructField(
        "Order UUID",
        StringType(),
        True
    ),
    StructField(
        "Coin Name",
        StringType(),
        True
    ),
    StructField(
        "Symbol",
        StringType(),
        True
    ),
    StructField(
        "Transaction Type",
        StringType(),
        True
    ),
    StructField(
        "Execution price (INR)",
        DecimalType(38, 18),
        True
    ),
    StructField(
        "Coin Quantity",
        DecimalType(38, 18),
        True
    ),
    StructField(
        "Order Amount (INR)",
        DecimalType(38, 18),
        True
    ),
    StructField(
        "Fees (including 18% GST) (INR)",
        DecimalType(38, 18),
        True
    ),
    StructField(
        "TDS (INR)",
        DecimalType(38, 18),
        True
    ),
    StructField(
        "Net Amount (INR)",
        DecimalType(38, 18),
        True
    )
])


inr_spot_raw_df = (
    spark.read
    .schema(inr_spot_schema)
    .option("multiline", True)
    .option("recursiveFileLookup", True)
    .json(
        f"{raw_root}/inr_spot_trades/"
    )
)


inr_spot_bronze_df = (
    inr_spot_raw_df
    .select(
        col("Investment Product")
        .alias("investment_product"),

        col("Date (IST)")
        .alias("date_ist"),

        col("Order UUID")
        .alias("order_id"),

        col("Coin Name")
        .alias("coin_name"),

        col("Symbol")
        .alias("symbol"),

        col("Transaction Type")
        .alias("transaction_type"),

        col("Execution price (INR)")
        .alias("execution_price_inr"),

        col("Coin Quantity")
        .alias("coin_quantity"),

        col("Order Amount (INR)")
        .alias("order_amount_inr"),

        col("Fees (including 18% GST) (INR)")
        .alias("fee_inr"),

        col("TDS (INR)")
        .alias("tds_inr"),

        col("Net Amount (INR)")
        .alias("net_amount_inr"),

        col("_metadata.file_name")
        .alias("source_file_name"),

        col("_metadata.file_path")
        .alias("_source_file")
    )
    .withColumn(
        "exchange",
        lit("Mudrex")
    )
    .withColumn(
        "source_type",
        lit("INR Spot Trade")
    )
    .withColumn(
        "ingested_at",
        current_timestamp()
    )
    .dropDuplicates([
        "order_id"
    ])
)


print(
    "Mudrex INR Spot rows:",
    inr_spot_bronze_df.count()
)


(
    inr_spot_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        True
    )
    .saveAsTable(
        f"{catalog}.{bronze_schema}.mudrex_inr_spot_trades"
    )
)


print(
    "Saved Mudrex INR Spot rows:",
    spark.table(
        f"{catalog}.{bronze_schema}.mudrex_inr_spot_trades"
    ).count()
)

Mudrex INR Spot rows: 0
Saved Mudrex INR Spot rows: 0


####Mudrex deposits

In [0]:
deposits_schema = StructType([
    StructField(
        "Transaction Type",
        StringType(),
        True
    ),
    StructField(
        "Date (IST)",
        StringType(),
        True
    ),
    StructField(
        "Deposit Type",
        StringType(),
        True
    ),
    StructField(
        "Fiat Amount",
        StringType(),
        True
    ),
    StructField(
        "Symbol",
        StringType(),
        True
    ),
    StructField(
        "Crypto Quantity",
        StringType(),
        True
    ),
    StructField(
        "Transaction ID",
        StringType(),
        True
    ),
    StructField(
        "Transaction Hash",
        StringType(),
        True
    ),
    StructField(
        "Transaction Value",
        StringType(),
        True
    )
])


deposits_raw_df = (
    spark.read
    .schema(deposits_schema)
    .option("multiline", True)
    .option("recursiveFileLookup", True)
    .json(
        f"{raw_root}/deposits/"
    )
)


deposits_bronze_df = (
    deposits_raw_df
    .select(
        col("Transaction Type")
        .alias("transaction_type"),

        col("Date (IST)")
        .alias("date_ist"),

        col("Deposit Type")
        .alias("deposit_type"),

        col("Fiat Amount")
        .alias("fiat_amount"),

        col("Symbol")
        .alias("symbol"),

        col("Crypto Quantity")
        .alias("crypto_quantity"),

        col("Transaction ID")
        .alias("transaction_id"),

        col("Transaction Hash")
        .alias("transaction_hash"),

        col("Transaction Value")
        .alias("transaction_value"),

        col("_metadata.file_name")
        .alias("source_file_name"),

        col("_metadata.file_path")
        .alias("_source_file")
    )
    .withColumn(
        "exchange",
        lit("Mudrex")
    )
    .withColumn(
        "source_type",
        lit("Deposit")
    )
    .withColumn(
        "ingested_at",
        current_timestamp()
    )
    .dropDuplicates([
        "transaction_id"
    ])
)


print(
    "Mudrex deposit rows:",
    deposits_bronze_df.count()
)

display(
    deposits_bronze_df
    .orderBy(
        to_timestamp(
            col("date_ist"),
            "MMMM d, yyyy HH:mm:ss"
        ).desc()
    )
)


(
    deposits_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        True
    )
    .saveAsTable(
        f"{catalog}.{bronze_schema}.mudrex_deposits"
    )
)


print(
    "Saved Mudrex deposit rows:",
    spark.table(
        f"{catalog}.{bronze_schema}.mudrex_deposits"
    ).count()
)

Mudrex deposit rows: 14


transaction_type,date_ist,deposit_type,fiat_amount,symbol,crypto_quantity,transaction_id,transaction_hash,transaction_value,source_file_name,_source_file,exchange,source_type,ingested_at
DEPOSIT,"March 13, 2026 19:15:01",Crypto,NA,SOL,0.00641492,019ce771-803c-7753-8c77-dae1d3bc059d,2qLcZAvFqTytyA9cYBBreQRM9AzJoPjqrkSKY88rJhb2UYKHmSzM4LttjjNz5PZUo1AN5tSekaTf1wFQ9U9u8e4M,0.5906216844 USDT,deposits_20260927T153314_975300Z_2a7ac6b8.json,s3://l1l2bucket/data/raw/mudrex/deposits/deposits_20260927T153314_975300Z_2a7ac6b8.json,Mudrex,Deposit,2026-09-28T15:32:38.933Z
DEPOSIT,"March 13, 2026 19:14:12",Fiat,4000,USDT,42.756375474768,019ce770-beac-76af-ac5b-495e6d9c14be,null,42.756375474768 USDT,deposits_20260927T153314_975300Z_2a7ac6b8.json,s3://l1l2bucket/data/raw/mudrex/deposits/deposits_20260927T153314_975300Z_2a7ac6b8.json,Mudrex,Deposit,2026-09-28T15:32:38.933Z
DEPOSIT,"March 13, 2026 19:10:02",Crypto,NA,SOL,0.0999999,019ce76c-ee3f-7118-9855-f17505cfe521,2eMTSsX64vspd93GeSffoL84BBo8pvfDwRNpD51446sHFZH3SEpD4rg3wJJa7euTh7qwUkRY33BJgbnwyujWBYv6,9.160990839 USDT,deposits_20260927T153314_975300Z_2a7ac6b8.json,s3://l1l2bucket/data/raw/mudrex/deposits/deposits_20260927T153314_975300Z_2a7ac6b8.json,Mudrex,Deposit,2026-09-28T15:32:38.933Z
DEPOSIT,"February 18, 2026 16:51:28",Fiat,5006,USDT,52.708818813463736,019c707b-cd77-705f-b55c-405f89f80782,null,52.708818813463736 USDT,deposits_20260927T153314_975300Z_2a7ac6b8.json,s3://l1l2bucket/data/raw/mudrex/deposits/deposits_20260927T153314_975300Z_2a7ac6b8.json,Mudrex,Deposit,2026-09-28T15:32:38.933Z
DEPOSIT,"February 10, 2026 13:49:40",Fiat,1200,USDT,12.3124999999992,019c46a2-7f08-79c7-8e1c-679f9ed7d73b,null,12.3124999999992 USDT,deposits_20260927T153314_975300Z_2a7ac6b8.json,s3://l1l2bucket/data/raw/mudrex/deposits/deposits_20260927T153314_975300Z_2a7ac6b8.json,Mudrex,Deposit,2026-09-28T15:32:38.933Z
DEPOSIT,"February 10, 2026 13:42:04",Fiat,1200,INR,NA,019c469b-8982-730d-a38f-3613fed7dfe5,null,1200 INR,deposits_20260927T153314_975300Z_2a7ac6b8.json,s3://l1l2bucket/data/raw/mudrex/deposits/deposits_20260927T153314_975300Z_2a7ac6b8.json,Mudrex,Deposit,2026-09-28T15:32:38.933Z
DEPOSIT,"February 1, 2026 00:41:39",Fiat,1000,USDT,10.456475583863,019c1577-cd7c-7010-8b5c-aecb6b1d5982,null,10.456475583863 USDT,deposits_20260927T153314_975300Z_2a7ac6b8.json,s3://l1l2bucket/data/raw/mudrex/deposits/deposits_20260927T153314_975300Z_2a7ac6b8.json,Mudrex,Deposit,2026-09-28T15:32:38.933Z
DEPOSIT,"November 24, 2025 03:28:57",Fiat,1002,USDT,10.608153681963037,019ab2ba-2ec6-7c74-b3a7-38c69983fbec,null,10.608153681963037 USDT,deposits_20260927T153314_975300Z_2a7ac6b8.json,s3://l1l2bucket/data/raw/mudrex/deposits/deposits_20260927T153314_975300Z_2a7ac6b8.json,Mudrex,Deposit,2026-09-28T15:32:38.933Z
DEPOSIT,"November 5, 2025 08:12:23",Fiat,503,USDT,5.453289617485429,019a51e4-d65e-7aad-9c0b-183cbf285eb1,null,5.453289617485429 USDT,deposits_20260927T153314_975300Z_2a7ac6b8.json,s3://l1l2bucket/data/raw/mudrex/deposits/deposits_20260927T153314_975300Z_2a7ac6b8.json,Mudrex,Deposit,2026-09-28T15:32:38.933Z
DEPOSIT,"October 11, 2025 15:26:06",Fiat,503,USDT,5.208517745302337,0199d2b2-eee0-7807-9910-57c54b22a02e,null,5.208517745302337 USDT,deposits_20260927T153314_975300Z_2a7ac6b8.json,s3://l1l2bucket/data/raw/mudrex/deposits/deposits_20260927T153314_975300Z_2a7ac6b8.json,Mudrex,Deposit,2026-09-28T15:32:38.933Z


Saved Mudrex deposit rows: 14


####Mudrex withdrawals

In [0]:
withdrawals_schema = StructType([
    StructField(
        "Transaction Type",
        StringType(),
        True
    ),
    StructField(
        "Date (IST)",
        StringType(),
        True
    ),
    StructField(
        "Withdraw Type",
        StringType(),
        True
    ),
    StructField(
        "Fiat Amount",
        StringType(),
        True
    ),
    StructField(
        "Symbol",
        StringType(),
        True
    ),
    StructField(
        "Crypto Quantity",
        StringType(),
        True
    ),
    StructField(
        "Fees (including 18% GST)",
        StringType(),
        True
    ),
    StructField(
        "TDS (INR)",
        StringType(),
        True
    ),
    StructField(
        "Net Crypto Quantity",
        StringType(),
        True
    ),
    StructField(
        "Transaction ID",
        StringType(),
        True
    ),
    StructField(
        "Transaction Hash",
        StringType(),
        True
    ),
    StructField(
        "Transaction Value",
        StringType(),
        True
    )
])


withdrawals_raw_df = (
    spark.read
    .schema(withdrawals_schema)
    .option("multiline", True)
    .option("recursiveFileLookup", True)
    .json(
        f"{raw_root}/withdrawals/"
    )
)


withdrawals_bronze_df = (
    withdrawals_raw_df
    .select(
        col("Transaction Type")
        .alias("transaction_type"),

        col("Date (IST)")
        .alias("date_ist"),

        col("Withdraw Type")
        .alias("withdraw_type"),

        col("Fiat Amount")
        .alias("fiat_amount"),

        col("Symbol")
        .alias("symbol"),

        col("Crypto Quantity")
        .alias("crypto_quantity"),

        col("Fees (including 18% GST)")
        .alias("fee"),

        col("TDS (INR)")
        .alias("tds_inr"),

        col("Net Crypto Quantity")
        .alias("net_crypto_quantity"),

        col("Transaction ID")
        .alias("transaction_id"),

        col("Transaction Hash")
        .alias("transaction_hash"),

        col("Transaction Value")
        .alias("transaction_value"),

        col("_metadata.file_name")
        .alias("source_file_name"),

        col("_metadata.file_path")
        .alias("_source_file")
    )
    .withColumn(
        "exchange",
        lit("Mudrex")
    )
    .withColumn(
        "source_type",
        lit("Withdrawal")
    )
    .withColumn(
        "ingested_at",
        current_timestamp()
    )
    .dropDuplicates([
        "transaction_id"
    ])
)


print(
    "Mudrex withdrawal rows:",
    withdrawals_bronze_df.count()
)

display(
    withdrawals_bronze_df
    .orderBy(
        to_timestamp(
            col("date_ist"),
            "MMMM d, yyyy HH:mm:ss"
        ).desc()
    )
)


(
    withdrawals_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        True
    )
    .saveAsTable(
        f"{catalog}.{bronze_schema}.mudrex_withdrawals"
    )
)


print(
    "Saved Mudrex withdrawal rows:",
    spark.table(
        f"{catalog}.{bronze_schema}.mudrex_withdrawals"
    ).count()
)

Mudrex withdrawal rows: 1


transaction_type,date_ist,withdraw_type,fiat_amount,symbol,crypto_quantity,fee,tds_inr,net_crypto_quantity,transaction_id,transaction_hash,transaction_value,source_file_name,_source_file,exchange,source_type,ingested_at
Withdraw,"February 10, 2026 13:48:56",Fiat,1200,INR,NA,0 INR,0,NA,019c46a1-d206-78c8-8404-4e7b19c06643,null,1200 INR,withdrawals_20260927T153315_117465Z_d5f5312e.json,s3://l1l2bucket/data/raw/mudrex/withdrawals/withdrawals_20260927T153315_117465Z_d5f5312e.json,Mudrex,Withdrawal,2026-09-28T15:36:25.811Z


Saved Mudrex withdrawal rows: 1


####Mudrex Earn history

In [0]:
earn_schema = StructType([
    StructField(
        "Symbol",
        StringType(),
        True
    ),
    StructField(
        "Coin Name",
        StringType(),
        True
    ),
    StructField(
        "Quantity",
        DecimalType(38, 18),
        True
    ),
    StructField(
        "Transaction Type",
        StringType(),
        True
    ),
    StructField(
        "Status",
        StringType(),
        True
    ),
    StructField(
        "Date (IST)",
        StringType(),
        True
    )
])


earn_raw_df = (
    spark.read
    .schema(earn_schema)
    .option("multiline", True)
    .option("recursiveFileLookup", True)
    .json(
        f"{raw_root}/earn_history/"
    )
)


earn_bronze_df = (
    earn_raw_df
    .select(
        col("Symbol")
        .alias("symbol"),

        col("Coin Name")
        .alias("coin_name"),

        col("Quantity")
        .alias("quantity"),

        col("Transaction Type")
        .alias("transaction_type"),

        col("Status")
        .alias("status"),

        col("Date (IST)")
        .alias("date_ist"),

        col("_metadata.file_name")
        .alias("source_file_name"),

        col("_metadata.file_path")
        .alias("_source_file")
    )
    .withColumn(
        "exchange",
        lit("Mudrex")
    )
    .withColumn(
        "source_type",
        lit("Earn History")
    )
    .withColumn(
        "ingested_at",
        current_timestamp()
    )
    .dropDuplicates([
        "date_ist",
        "symbol",
        "transaction_type",
        "quantity"
    ])
)


print(
    "Mudrex Earn rows:",
    earn_bronze_df.count()
)

display(
    earn_bronze_df
    .orderBy(
        to_timestamp(
            col("date_ist"),
            "MMMM d, yyyy HH:mm:ss"
        ).desc()
    )
)


(
    earn_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        True
    )
    .saveAsTable(
        f"{catalog}.{bronze_schema}.mudrex_earn_history"
    )
)


print(
    "Saved Mudrex Earn rows:",
    spark.table(
        f"{catalog}.{bronze_schema}.mudrex_earn_history"
    ).count()
)

Mudrex Earn rows: 120


symbol,coin_name,quantity,transaction_type,status,date_ist,source_file_name,_source_file,exchange,source_type,ingested_at
SOL,Solana,0.000519650000000000,CREDIT,Successful,"March 29, 2026 17:30:21",earn_history_20260927T153315_259708Z_dea303e7.json,s3://l1l2bucket/data/raw/mudrex/earn_history/earn_history_20260927T153315_259708Z_dea303e7.json,Mudrex,Earn History,2026-09-28T15:37:10.692Z
SOL,Solana,0.000519650000000000,INTEREST,Successful,"March 29, 2026 17:30:04",earn_history_20260927T153315_259708Z_dea303e7.json,s3://l1l2bucket/data/raw/mudrex/earn_history/earn_history_20260927T153315_259708Z_dea303e7.json,Mudrex,Earn History,2026-09-28T15:37:10.692Z
ETH,Ethereum,0.000011970000000000,INTEREST,Successful,"March 29, 2026 17:29:31",earn_history_20260927T153315_259708Z_dea303e7.json,s3://l1l2bucket/data/raw/mudrex/earn_history/earn_history_20260927T153315_259708Z_dea303e7.json,Mudrex,Earn History,2026-09-28T15:37:10.692Z
SUI,Sui,13.936812720000000000,CREDIT,Successful,"March 29, 2026 17:29:14",earn_history_20260927T153315_259708Z_dea303e7.json,s3://l1l2bucket/data/raw/mudrex/earn_history/earn_history_20260927T153315_259708Z_dea303e7.json,Mudrex,Earn History,2026-09-28T15:37:10.692Z
ETH,Ethereum,0.000001480000000000,INTEREST,Successful,"March 15, 2026 22:44:03",earn_history_20260927T153315_259708Z_dea303e7.json,s3://l1l2bucket/data/raw/mudrex/earn_history/earn_history_20260927T153315_259708Z_dea303e7.json,Mudrex,Earn History,2026-09-28T15:37:10.692Z
BTC,Bitcoin,0.000094120000000000,CREDIT,Successful,"March 13, 2026 19:31:57",earn_history_20260927T153315_259708Z_dea303e7.json,s3://l1l2bucket/data/raw/mudrex/earn_history/earn_history_20260927T153315_259708Z_dea303e7.json,Mudrex,Earn History,2026-09-28T15:37:10.692Z
BTC,Bitcoin,1.0000000000E-8,INTEREST,Successful,"March 13, 2026 19:31:43",earn_history_20260927T153315_259708Z_dea303e7.json,s3://l1l2bucket/data/raw/mudrex/earn_history/earn_history_20260927T153315_259708Z_dea303e7.json,Mudrex,Earn History,2026-09-28T15:37:10.692Z
SOL,Solana,0.141554860000000000,CREDIT,Successful,"March 13, 2026 19:31:30",earn_history_20260927T153315_259708Z_dea303e7.json,s3://l1l2bucket/data/raw/mudrex/earn_history/earn_history_20260927T153315_259708Z_dea303e7.json,Mudrex,Earn History,2026-09-28T15:37:10.692Z
SOL,Solana,0.000250150000000000,INTEREST,Successful,"March 13, 2026 19:31:24",earn_history_20260927T153315_259708Z_dea303e7.json,s3://l1l2bucket/data/raw/mudrex/earn_history/earn_history_20260927T153315_259708Z_dea303e7.json,Mudrex,Earn History,2026-09-28T15:37:10.692Z
ETH,Ethereum,0.003177210000000000,CREDIT,Successful,"March 13, 2026 19:30:57",earn_history_20260927T153315_259708Z_dea303e7.json,s3://l1l2bucket/data/raw/mudrex/earn_history/earn_history_20260927T153315_259708Z_dea303e7.json,Mudrex,Earn History,2026-09-28T15:37:10.692Z


Saved Mudrex Earn rows: 120


####Mudrex USDT fund transactions

In [0]:
usdt_funds_schema = StructType([
    StructField(
        "Date (IST)",
        StringType(),
        True
    ),
    StructField(
        "Order UUID",
        StringType(),
        True
    ),
    StructField(
        "Symbol",
        StringType(),
        True
    ),
    StructField(
        "Transaction Type",
        StringType(),
        True
    ),
    StructField(
        "Conversion Rate",
        DecimalType(38, 18),
        True
    ),
    StructField(
        "Quantity",
        DecimalType(38, 18),
        True
    ),
    StructField(
        "Order Amount (INR)",
        DecimalType(38, 18),
        True
    ),
    StructField(
        "Fees (including 18% GST) (INR)",
        DecimalType(38, 18),
        True
    ),
    StructField(
        "TDS (INR)",
        DecimalType(38, 18),
        True
    ),
    StructField(
        "Net Amount (INR)",
        DecimalType(38, 18),
        True
    )
])


usdt_funds_raw_df = (
    spark.read
    .schema(usdt_funds_schema)
    .option("multiline", True)
    .option("recursiveFileLookup", True)
    .json(
        f"{raw_root}/usdt_fund_transactions/"
    )
)


usdt_funds_bronze_df = (
    usdt_funds_raw_df
    .select(
        col("Date (IST)")
        .alias("date_ist"),

        col("Order UUID")
        .alias("order_id"),

        col("Symbol")
        .alias("symbol"),

        col("Transaction Type")
        .alias("transaction_type"),

        col("Conversion Rate")
        .alias("conversion_rate"),

        col("Quantity")
        .alias("quantity"),

        col("Order Amount (INR)")
        .alias("order_amount_inr"),

        col("Fees (including 18% GST) (INR)")
        .alias("fee_inr"),

        col("TDS (INR)")
        .alias("tds_inr"),

        col("Net Amount (INR)")
        .alias("net_amount_inr"),

        col("_metadata.file_name")
        .alias("source_file_name"),

        col("_metadata.file_path")
        .alias("_source_file")
    )
    .withColumn(
        "exchange",
        lit("Mudrex")
    )
    .withColumn(
        "source_type",
        lit("USDT Fund Transaction")
    )
    .withColumn(
        "ingested_at",
        current_timestamp()
    )
    .dropDuplicates([
        "order_id"
    ])
)


print(
    "Mudrex USDT fund rows:",
    usdt_funds_bronze_df.count()
)


(
    usdt_funds_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        True
    )
    .saveAsTable(
        f"{catalog}.{bronze_schema}.mudrex_usdt_fund_transactions"
    )
)


print(
    "Saved Mudrex USDT fund rows:",
    spark.table(
        f"{catalog}.{bronze_schema}.mudrex_usdt_fund_transactions"
    ).count()
)

Mudrex USDT fund rows: 0
Saved Mudrex USDT fund rows: 0


In [0]:
mudrex_bronze_tables = [
    "mudrex_usdt_spot_trades",
    "mudrex_inr_spot_trades",
    "mudrex_deposits",
    "mudrex_withdrawals",
    "mudrex_earn_history",
    "mudrex_usdt_fund_transactions"
]


for table_name in mudrex_bronze_tables:

    row_count = spark.table(
        f"{catalog}.{bronze_schema}.{table_name}"
    ).count()

    print(
        f"{table_name}: {row_count} rows"
    )

mudrex_usdt_spot_trades: 37 rows
mudrex_inr_spot_trades: 0 rows
mudrex_deposits: 14 rows
mudrex_withdrawals: 1 rows
mudrex_earn_history: 120 rows
mudrex_usdt_fund_transactions: 0 rows
